# Dropout/discontinuation reason harmonization investigation

**Question:** `disposit.sas7bdat`'s `DSREAS`/`DSREASCD` (discontinuation reason) is present in
`NCT00339183` and `NCT00364013` but **missing entirely** in `NCT00115225` (PACCE). PACCE only has
`EOIP`/`EOIPCD`/`EOIPDY` (shared by all 3 trials). Is `EOIP` a real substitute for `DSREAS`,
or something coarser?

**Finding:** `EOIP`/`EOIPCD` is present in all 3 trials with the *same category labels* as
`DSREAS` (Adverse event, Subject request, Administrative decision, Consent withdrawn,
Noncompliance, Death, Other) — it's the same underlying concept, not a degraded proxy.

**Wrinkle:** PACCE's `EOIP` has **no 'Disease progression' category at all**, unlike the other
two trials. That's suspicious for an oncology trial — checked whether PACCE tracks progression
elsewhere, and it does, in `a_eendpt`/`a_sendpt` (efficacy/survival endpoint domains), separate
from the admin disposition domain. **Caution:** the obvious-looking `PDYN` column is a false
friend — its real label is `Protocol Deviation?`, not progression. The actual progression fields,
confirmed via SAS column labels, are `PDCR` (`PD (Central, RECIST) on Study` — 1/0 event
indicator) paired with `PDDYCR` (`PD Day (Central, RECIST)`) — populated for every patient
(event day if `PDCR==1`, otherwise the censoring day). **All 3 trials have this identical
`PDCR`/`PDDYCR` pair in `a_eendpt`**, same label, same 1/0 coding — not PACCE-specific.

**Recommendation:** harmonize on two fields, not one:
1. **Discontinuation reason** — `EOIP`/`EOIPCD` from `disposit`, same schema across all 3 trials.
2. **Disease progression event/day** — `PDCR`/`PDDYCR` from `a_eendpt`, pulled the
   *same way for all 3 trials* (not just PACCE) — standardizes on one measurement method
   (central RECIST review) instead of each trial's own inconsistent admin-reason text label.

This keeps all 3 raw-domain trials (2,369+ patients) in the pooled dataset, per the spec's N target.


In [1]:
import pyreadstat
from pathlib import Path
import pandas as pd

pd.set_option('display.max_rows', None)

DATA_DIR = Path('..') / '..' / 'Data' / 'raw'

TRIALS = {
    'PACCE (NCT00115225)': 'NCT00115225_PACCE_bev_panitumumab_raw',
    'FOLFIRI (NCT00339183)': 'NCT00339183_panitumumab_folfiri_raw',
    'FOLFOX/PRIME (NCT00364013)': 'NCT00364013_panitumumab_folfox_raw',
}


## 1. `disposit` — EOIP vs DSREAS value comparison

**What these fields mean:**

- **`EOIP` / `EOIPCD`** — "End of Investigational Product" (reason / reason code). Why the patient
  stopped taking the study drug — the coded reason categories are things like Adverse event,
  Subject request, Administrative decision, Disease progression, Death, etc. This is PACCE's
  version of a discontinuation-reason field.
- **`DSREAS` / `DSREASCD`** — "Disposition Reason" (text / code). The CDISC-conventional name for
  the same concept in the other two trials — same idea as `EOIP`, different variable name because
  PACCE's export doesn't follow the same domain-naming convention.
- The `CD` suffix on both just means "coded" — a numeric code paired with the human-readable text
  version of the same field (e.g. `EOIPCD == 5.0` corresponds to `EOIP == 'Adverse event'`).

Below: value counts for all four, across all 3 trials, so you can see the category overlap directly
rather than just the presence/absence table from `column_comparison.ipynb`.

In [2]:
for label, folder in TRIALS.items():
    path = DATA_DIR / folder / 'disposit.sas7bdat'
    df, meta = pyreadstat.read_sas7bdat(str(path))
    print(f'===== {label} — n={len(df)} =====')
    for col in ['EOIP', 'EOIPCD', 'DSREAS', 'DSREASCD']:
        if col in df.columns:
            print(f'-- {col}:')
            print(df[col].value_counts(dropna=False))
        else:
            print(f'-- {col}: NOT PRESENT')
        print()


===== PACCE (NCT00115225) — n=842 =====
-- EOIP:
EOIP
                      420
Other                 221
Adverse event         128
Subject request        65
Protocol deviation      8
Name: count, dtype: int64

-- EOIPCD:
EOIPCD
NaN     420
88.0    221
5.0     128
13.0     65
3.0       8
Name: count, dtype: int64

-- DSREAS: NOT PRESENT

-- DSREASCD: NOT PRESENT

===== FOLFIRI (NCT00339183) — n=946 =====
-- EOIP:
EOIP
                           475
Disease progression        286
Adverse event               72
Subject request             59
Death                       16
Consent withdrawn           13
Administrative decision     11
Other                        8
Noncompliance                3
Protocol deviation           2
Lost to follow-up            1
Name: count, dtype: int64

-- EOIPCD:
EOIPCD
NaN     475
7.0     286
5.0      72
13.0     59
11.0     16
6.0      13
9.0      11
88.0      8
4.0       3
3.0       2
10.0      1
Name: count, dtype: int64

-- DSREAS:
DSREAS
               

## 2. `a_eendpt`/`a_sendpt` — progression fields, all 3 trials

**What these domains are:**

- **`a_eendpt`** — "analysis efficacy endpoint" domain: derived variables for whether/when a
  patient's tumor progressed, responded, etc. — the efficacy side of the trial, separate from
  administrative disposition (`disposit`).
- **`a_sendpt`** — "analysis survival endpoint" domain: derived variables for overall survival
  (death) and related time-to-event summaries.
- Both are *derived/analysis* domains (the `a_` prefix), not raw source data — meaning someone at
  the sponsor already computed these from underlying tumor assessment visits, so we get
  ready-to-use day/event fields instead of having to derive them ourselves from raw scan dates.

The search below just greps column names for `PD`/`PROG` to find candidate progression fields —
a first pass only. Section 3 below double-checks the real meaning via the SAS column label,
since `PD` turned out to be an ambiguous prefix (see the `PDYN` catch).

In [3]:
for label, folder in TRIALS.items():
    print(f'===== {label} =====')
    for fname in ['a_eendpt.sas7bdat', 'a_sendpt.sas7bdat']:
        _, meta = pyreadstat.read_sas7bdat(str(DATA_DIR / folder / fname), metadataonly=True)
        cols = list(meta.column_names)
        prog_like = [c for c in cols if any(k in c.upper() for k in ['PD', 'PROG'])]
        print(f'  {fname}: {prog_like}')
    print()


===== PACCE (NCT00115225) =====
  a_eendpt.sas7bdat: ['PDADEN', 'PDICSIG', 'PDICMIS', 'PDPROMED', 'PDYN', 'PDDYCR', 'PDCR', 'PDMTCR', 'PDDYLR', 'PDMTLR', 'PDLR', 'PDCR30', 'PDDYCR30', 'PDMTCR30', 'PDLR30', 'PDDYLR30', 'PDMTLR30', 'PDCRND', 'PDDYCRND', 'PDMTCRND']
  a_sendpt.sas7bdat: ['PDADEN', 'PDICSIG', 'PDICMIS', 'PDPROMED', 'PDYN', 'IPDURWK', 'IPDSINT', 'IPDSINTR', 'IPDLYPC', 'IPDECPCW']

===== FOLFIRI (NCT00339183) =====
  a_eendpt.sas7bdat: ['PDDYCR', 'PDCR', 'PDOCR', 'PDDYOCR', 'PDOLR', 'PDDYOLR', 'PDDYLR', 'PDLR']
  a_sendpt.sas7bdat: ['IPDURWK', 'IPDSINT', 'IPDSINTR', 'IPDSINR3', 'IPDLYNM', 'IPDLYPC', 'IPDECNMW', 'IPDECPCW', 'PROPDOSE']

===== FOLFOX/PRIME (NCT00364013) =====
  a_eendpt.sas7bdat: ['PDDYCR', 'PDCR', 'PDOCR', 'PDDYOCR', 'PDOLR', 'PDDYOLR', 'PDDYLR', 'PDLR']
  a_sendpt.sas7bdat: ['IPDURWK', 'IPDSINT', 'IPDSINTR', 'IPDSINR3', 'IPDLYNM', 'IPDLYPC', 'IPDECNMW', 'IPDECPCW', 'PROPDOSE']



## 3. `PDCR` / `PDDYCR` — confirm the event/day pair behaves consistently across all 3 trials

Note: `PDYN` looked like the obvious progression flag by name, but its real SAS column label is
`Protocol Deviation?` — not progression. Checked labels explicitly below before trusting anything.

**What `PDCR`/`PDDYCR` actually mean, confirmed via SAS column labels:**

- **`PDCR`** — "PD (Central, RECIST) on Study" — a 1/0 flag for whether the patient's disease
  progressed **on study**, as determined by **central** review (an independent radiology panel
  re-reading the scans) using **RECIST** (Response Evaluation Criteria in Solid Tumors — the
  standard rulebook oncology trials use to define "progression" from tumor measurements).
  `1` = progressed, `0` = censored (no progression observed by data cutoff/last assessment).
- **`PDDYCR`** — "PD Day (Central, RECIST)" — the day number paired with `PDCR`. When
  `PDCR == 1` this is the day progression was confirmed; when `PDCR == 0` it's the day of the
  patient's last evaluable assessment (the censoring day) — which is why it's populated for
  *every* patient, not just the ones who progressed. This dual meaning (event day vs. censoring
  day, disambiguated by the paired status flag) is the standard time-to-event data shape you'll
  need to reproduce for the harmonized outcome variable in `dbt`.
- **"Central" vs. "Investigator" (`PDDYLR` = local/investigator-read)** — trials record tumor
  progression two ways: the treating site's own radiologist (investigator/local read, "LR") and
  an independent, blinded central review ("CR"). Central review is the less biased, more
  rigorous version — generally the preferred field for the harmonized schema.
- Sanity-checked with `.describe()` split by `PDCR` below: the `PDCR==1` group's days should
  behave like real event times, the `PDCR==0` group's like censoring times — worth eyeballing
  before trusting the field for the actual Cox/RSF fit later.

In [4]:
for label, folder in TRIALS.items():
    path = DATA_DIR / folder / 'a_eendpt.sas7bdat'
    df, meta = pyreadstat.read_sas7bdat(str(path))

    # confirm the column label before trusting it (PDYN taught us not to trust names alone)
    label_lookup = dict(zip(meta.column_names, meta.column_labels))
    print(f'===== {label} — a_eendpt, n={len(df)} =====')
    print(f"PDCR label:    {label_lookup.get('PDCR')}")
    print(f"PDDYCR label:  {label_lookup.get('PDDYCR')}")
    print()
    print('PDCR value_counts (event=1, censored=0):')
    print(df['PDCR'].value_counts(dropna=False))
    print()
    print('PDDYCR non-null:', df['PDDYCR'].notna().sum(), '/', len(df))
    print('PDDYCR describe, split by PDCR:')
    print(df.groupby('PDCR')['PDDYCR'].describe())
    print()


===== PACCE (NCT00115225) — a_eendpt, n=842 =====
PDCR label:    PD (Central, RECIST) on Study
PDDYCR label:  PD Day (Central, RECIST)

PDCR value_counts (event=1, censored=0):
PDCR
1.0    582
0.0    260
Name: count, dtype: int64

PDDYCR non-null: 842 / 842
PDDYCR describe, split by PDCR:
      count        mean         std  min    25%    50%     75%     max
PDCR                                                                  
0.0   260.0  220.473077  208.715235  1.0    1.0  169.5  351.00   868.0
1.0   582.0  367.197595  246.095235  7.0  184.0  310.5  492.75  1286.0

===== FOLFIRI (NCT00339183) — a_eendpt, n=946 =====
PDCR label:    PD (Central, RECIST) on Study
PDDYCR label:  PD Day (Central, RECIST)

PDCR value_counts (event=1, censored=0):
PDCR
1.0    648
0.0    298
Name: count, dtype: int64

PDDYCR non-null: 946 / 946
PDDYCR describe, split by PDCR:
      count        mean         std  min   25%    50%     75%     max
PDCR                                                           